In [ ]:
from src.entities.tubular_load_case import TubularLoadCase
from src.entities.steel_grade import SteelGrade
from src.entities.tubular import TubularData



pipe = TubularData(outer_diameter_in=10.75,
                   wall_thickness_in=0.545,
                   ovality_percent=0.6,
                   eccentricity_percent=0.0)

steel = SteelGrade("P110", 3e7, 0.3, 110e3, 125e3, 0.283, None)



from src.collapse.classic_mechanics import ClinedinstClassicalMethod
from src.collapse.api5c3_design import API5C3Method
from src.collapse.klever_tamano import KleverTamanoMethod

method = ClinedinstClassicalMethod()
Pe = method.calculate(tubular=pipe, material=steel)
print(Pe)
method = API5C3Method()
Pe = method.calculate(tubular=pipe, material=steel)
print(Pe)
method = KleverTamanoMethod()
Pe = method.calculate(tubular=pipe, material=steel)
print(Pe)




SteelGrade(name='P110', young_modulus_psi=30000000.0, poisson_ratio=0.3, yield_strength_psi=110000.0, ultimate_strength_psi=125000.0, density_lb_in3=0.283, stress_strain_curve=None)
CollapseResult(pressure_psi=np.float64(5828.620458643065), regime='Classical Mechanics', method='CLINEDINST_CLASSICAL_ELASTIC', metadata={'slenderness': 19.724770642201833, 'effective_yield_psi': 110000.0, 'parameters': {'Pe': 9533.774655503017, 'Py': 11153.488372093025, 'oval_factor': 1.355045871559633, 'inner_pressure': 0.0}})
CollapseResult(pressure_psi=5877.368142358142, regime='plastic', method='API_5C3', metadata={'slenderness': 19.724770642201833, 'effective_yield_psi': 110000.0, 'parameters': {'A': 3.180801308, 'B': 0.0819029, 'C': 2851.828059, 'F': 2.065664413941366, 'G': 0.053189083361820065}})
CollapseResult(pressure_psi=8470.119212624539, regime='elastic-plastic-transition', method='KLEVER_TAMANO', metadata={'slenderness': 19.724770642201833, 'effective_yield_psi': 110000.0, 'parameters': {'Pe':

In [3]:
def KleverTamano(pipe, grade, kels=1.089, kyls=0.991,
                 residualStress=0.0, kneedShape=False,
                 axialStress=0.0, innerPressure=0.0, **kwargs):
    """
    Legacy Klever-Tamano function (deprecated).

    Use KleverTamanoMethod class instead for new code.
    """
    hn = 0.017 if kneedShape is True else 0.0
    slen = pipe["OD"] / pipe["thickness"]

    Ht = 0.127 * pipe['ovality'] + 0.0039 * pipe['eccentricity'] - 0.44 * residualStress + hn
    Ht = max((0, Ht))

    Hy = 0
    He = 0
    # Elastic
    eta = 1 / (slen - 1)
    c = -1 + 1 / slen  # clinedinst
    E1 = kels * (1 - He) * grade["young"]
    Pe = 2 * (E1 / (1 - grade["poisson"]**2)) * (eta**3) * (1 + c * eta)

    # Yield
    Sy1 = kyls * (1 - Hy) * grade["yieldStress"]
    Si = (axialStress + innerPressure) / Sy1
    part1 = eta * Sy1 * (4 * (1 + 2 * eta)) / (3 + (1 + 2 * eta)**2)
    Part2_1 = -Si + (1 + 3 * (1 - Si**2) / ((1 + 2 * eta)**2))**0.5
    Part2_2 = -Si - (1 + 3 * (1 - Si**2) / ((1 + 2 * eta)**2))**0.5

    deltaPy1 = part1 * Part2_1
    deltaPy2 = part1 * Part2_2
    deltaPy = max(deltaPy1, deltaPy2)
    average = 0.5 * (deltaPy + 2 * eta * Sy1)

    Py = min((deltaPy, average))

    Pc = 2 * (Py * Pe) / (Py + Pe + ((Py - Pe)**2 + 4 * Ht * Py * Pe)**0.5)
    return Pc

print(KleverTamano(dict(OD=10.75, thickness=0.545, ovality=0.6, eccentricity=0.0), dict(young=3e7, poisson=0.3, yieldStress=110e3)))


8470.119212624539
